## Prepare questions from NIFD for testing GRPO model

Created by: Sahana Kowshik

Date: 05/07/2025

Modified: 11/01/2025

In [1]:
import pandas as pd
import json
import random

In [2]:
directory_path = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nifd"
test = pd.read_csv(f"{directory_path}/training_data/testing_data_grpo/test_summary.csv")

In [3]:
test.head()

,LONI_ID,CLINICAL_LINKDATE,DX,SITE,VISIT_NUMBER,DOB,GENDER,EDUCATION,RACE,CDR_DCDATE,...,MDEXAM_UPDRS,DOB_convert,CLINICAL_LINKDATE_convert,AGE,COHORT,patient_summary,diag_summary,brain_findings_summary,visit_summary_prompt,visit_summary
0,1_S_0119,6/18/14,BV,UCSF,4,12/17/49,2,16.0,1,6/17/14,...,20.0,1949-12-17,2014-06-18,64.501027,NIFD,"{""Subject Demographics"": {""Gender"": ""Female"", ...","{""Diagnosis"": ""Behavioral Variant FTD""}",Medial Parietal region has severe atrophy; Fro...,<|im_start|>user\nYou will receive patient dat...,The patient is a 64-year-old white female with...
1,2_S_0007,12/17/12,BV,MAYO,4,1/10/46,1,12.0,1,12/17/12,...,0.0,1946-01-10,2012-12-17,66.934976,NIFD,"{""Subject Demographics"": {""Gender"": ""Male"", ""Y...","{""Diagnosis"": ""Behavioral Variant FTD""}",NaN,<|im_start|>user\nYou will receive patient dat...,The subject is a 66-year-old White male with 1...
2,1_S_0059,11/19/13,BV,UCSF,3,8/27/65,2,16.0,1,11/18/13,...,11.0,1965-08-27,2013-11-19,48.229979,NIFD,"{""Subject Demographics"": {""Gender"": ""Female"", ...","{""Diagnosis"": ""Behavioral Variant FTD""}",Hippocampus region has moderate atrophy; Media...,<|im_start|>user\nYou will receive patient dat...,The patient is a 48-year-old White female with...
3,1_S_0248,8/27/15,CON,UCSF,5,1/1/40,1,18.0,1,NaN,...,NaN,1940-01-01,2015-08-27,75.652293,NIFD,"{""Subject Demographics"": {""Gender"": ""Male"", ""Y...","{""Diagnosis"": ""Healthy Control""}",NaN,<|im_start|>user\nYou will receive patient dat...,The patient is a 75-year-old White male with 1...
4,1_S_0203,9/8/15,BV,UCSF,3,9/17/52,1,20.0,1,9/8/15,...,0.0,1952-09-17,2015-09-08,62.973306,NIFD,"{""Subject Demographics"": {""Gender"": ""Male"", ""Y...","{""Diagnosis"": ""Behavioral Variant FTD""}",Medial Temporal region has moderate atrophy; L...,<|im_start|>user\nYou will receive patient dat...,The subject is a 62-year-old White male with 2...


In [4]:
print(test.iloc[0]['visit_summary'])

The patient is a 64-year-old white female with 16 years of education. On the Unified Parkinson's Disease Rating Scale (UPDRS), her score was 20, indicating some level of motor impairment associated with Parkinson’s disease. The Neuropsychiatric Inventory (Q Form) revealed the presence of depression with a severity score of 2, as well as elation or euphoria, apathy or indifference, disinhibition, irritability or lability, motor disturbance, nighttime behaviors, and appetite or eating concerns, each with varying severity levels ranging from 1 to 2. No delusions or hallucinations were reported, and anxiety was not present. The Geriatric Depression Scale (GDS) scores were 8 out of 30 and 5 out of 15, suggesting mild depressive symptoms. Regarding Activities of Daily Living, the Functional Activities Questionnaire indicated moderate impairment in tasks such as handling bills, taxes, and travel, while other activities showed mild difficulty. The Schwab and England Activities of Daily Living 

In [5]:
json.loads(test.iloc[0]['patient_summary'])

{'Subject Demographics': {'Gender': 'Female',
  'Years of Education': 16,
  'Race': 'White',
  'Age': 64},
 "Unified Parkinson's Disease Rating Scale (UPDRS)": {"MD Exam - Unified Parkinson's Disease Rating Scale": 20},
 'Neuropsychiatric Inventory': {'Neuropsychiatric Inventory - Q Form - Delusions': 'No',
  'Neuropsychiatric Inventory - Q Form - Hallucination Distress': 0,
  'Neuropsychiatric Inventory - Q Form - Agitation': 'No',
  'Neuropsychiatric Inventory - Q Form - Depression': 'Yes',
  'Neuropsychiatric Inventory - Q Form - Depression Severity': 2,
  'Neuropsychiatric Inventory - Q Form - Anxiety': 'No',
  'Neuropsychiatric Inventory - Q Form - Elation/Euphoria': 'Yes',
  'Neuropsychiatric Inventory - Q Form - Elation/Euphoria Severity': 2,
  'Neuropsychiatric Inventory - Q Form - Apathy/Indifference': 'Yes',
  'Neuropsychiatric Inventory - Q Form - Apathy/Indifference Severity': 1,
  'Neuropsychiatric Inventory - Q Form - Disinhibition': 'Yes',
  'Neuropsychiatric Inventory -

In [6]:
len(test)

292

In [7]:
columns = ['ID', 'patient_summary', 'diag_summary', 'visit_summary', 'question', 'options', 'ground_truth', 'ground_truth_text', 'COHORT']

# Add cognitive status question


In [8]:
import random, string

# Question variants for cognitive status identification
cog_question_variants = [
    "Using the information provided, determine the patient's cognitive status.",
    "Identify the patient's cognitive status based on the given information.",
    "Based on the provided clinical details, classify the patient's cognitive status.",
    "From the information available, determine the correct cognitive status for the patient.",
    "Analyze the patient's information and identify their cognitive status."
]

# Original answer texts
COG_OPTION_TEXTS = [
    "Normal Cognition (NC)",
    "Mild Cognitive Impairment (MCI)",
    "Dementia (DE)"
]

# Mapping from NACCUDSD to the correct label text
COG_ANSWER_MAP = {
    "CON": "Normal Cognition (NC)",
    "SV": "Dementia (DE)",
    "PNFA": "Dementia (DE)",
    "BV": "Dementia (DE)",
}

def add_cog_question(row, *, seed=None):
    rng = random.Random(seed if seed is not None else row.name)

    # 1️⃣ Random question phrasing
    row['question'] = rng.choice(cog_question_variants)

    # 2️⃣ Shuffle the answer options
    shuffled = COG_OPTION_TEXTS.copy()
    rng.shuffle(shuffled)
    letters = list(string.ascii_uppercase[:len(shuffled)])
    row['options'] = "\n".join(f"{ltr}. {opt}" for ltr, opt in zip(letters, shuffled))

    # 3️⃣ Determine correct answer letter
    correct_text = COG_ANSWER_MAP.get(row['DX'], None)
    if correct_text in shuffled:
        row['ground_truth'] = letters[shuffled.index(correct_text)]
    else:
        row['ground_truth'] = None  # fallback for unrecognized codes
    
    row['ground_truth_text'] = correct_text

    return row


In [9]:
# Apply the function to the MCI training set
test_cog = test.apply(add_cog_question, axis=1)

In [10]:
test_cog['ground_truth_text'].value_counts()

ground_truth_text
Dementia (DE)            156
Normal Cognition (NC)    136
Name: count, dtype: int64

In [11]:
test_cog["ID"] = test_cog["LONI_ID"]
test_cog = test_cog[columns]
print(len(test_cog))
test_cog = test_cog.dropna(how='any').reset_index(drop=True)
print(len(test_cog))

292
292


/scratch/ipykernel_223529/2283208384.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_cog["ID"] = test_cog["LONI_ID"]


In [12]:
test_cog['Q_TYPE'] = "Cognitive status"

In [13]:
test_cog.head()

,ID,patient_summary,diag_summary,visit_summary,question,options,ground_truth,ground_truth_text,COHORT,Q_TYPE
0,1_S_0119,"{""Subject Demographics"": {""Gender"": ""Female"", ...","{""Diagnosis"": ""Behavioral Variant FTD""}",The patient is a 64-year-old white female with...,"From the information available, determine the ...",A. Dementia (DE)\nB. Normal Cognition (NC)\nC....,A,Dementia (DE),NIFD,Cognitive status
1,2_S_0007,"{""Subject Demographics"": {""Gender"": ""Male"", ""Y...","{""Diagnosis"": ""Behavioral Variant FTD""}",The subject is a 66-year-old White male with 1...,Identify the patient's cognitive status based ...,A. Mild Cognitive Impairment (MCI)\nB. Normal ...,C,Dementia (DE),NIFD,Cognitive status
2,1_S_0059,"{""Subject Demographics"": {""Gender"": ""Female"", ...","{""Diagnosis"": ""Behavioral Variant FTD""}",The patient is a 48-year-old White female with...,"Using the information provided, determine the ...",A. Mild Cognitive Impairment (MCI)\nB. Dementi...,B,Dementia (DE),NIFD,Cognitive status
3,1_S_0248,"{""Subject Demographics"": {""Gender"": ""Male"", ""Y...","{""Diagnosis"": ""Healthy Control""}",The patient is a 75-year-old White male with 1...,Identify the patient's cognitive status based ...,A. Mild Cognitive Impairment (MCI)\nB. Normal ...,B,Normal Cognition (NC),NIFD,Cognitive status
4,1_S_0203,"{""Subject Demographics"": {""Gender"": ""Male"", ""Y...","{""Diagnosis"": ""Behavioral Variant FTD""}",The subject is a 62-year-old White male with 2...,Identify the patient's cognitive status based ...,A. Dementia (DE)\nB. Normal Cognition (NC)\nC....,A,Dementia (DE),NIFD,Cognitive status


In [14]:
test_cog.to_csv(f"{directory_path}/training_data/testing_data_grpo/with_summary/test_cog.csv", index=False)

# Add primary etiology question

In [15]:
import random, string

etiology_question_variants = [
    "Identify the primary etiologic diagnosis of the patient using the information provided, if applicable.",
    "Identify the primary etiology of the patient's cognitive impairment using the information provided, if applicable.",
    "Based on the clinical details, determine the main cause of the patient's cognitive impairment, if applicable.",
    "Using the information provided, identify the primary cause of the patient's cognitive decline, if applicable.",
    "Determine the primary etiology underlying the patient's cognitive impairment based on the provided information, if applicable."
]

# Raw etiology answer texts in original order
ETIOLOGY_OPTION_TEXTS = [
    "Alzheimer's disease (AD)",
    "Lewy body disease (LBD)",
    "Frontotemporal lobar degeneration and its variants, including primary progressive aphasia, corticobasal degeneration and progressive supranuclear palsy, and with or without amyotrophic lateral sclerosis (FTLD)",
    "Vascular brain injury or vascular dementia including stroke (VD)",
    "Systemic and environmental factors including infectious diseases (HIV included), metabolic, substance abuse / alcohol, medications, systemic disease and delirium (SEF)",
    "Psychiatric conditions including schizophrenia, depression, bipolar disorder, anxiety and posttraumatic stress disorder (PSY)",
    "Other (Multiple system atrophy, Essential tremor, Down syndrome, Huntington's disease, Prion disease, Traumatic brain injury, Normal-pressure hydrocephalus, Epilepsy, CNS neoplasm, etc)",
    "Not applicable (no cognitive impairment)",
]

# Mapping of ETPR code to correct answer text
ETPR_ANSWER_MAP = {
    'BV': ETIOLOGY_OPTION_TEXTS[2],
    'PNFA': ETIOLOGY_OPTION_TEXTS[2],
    'SV': ETIOLOGY_OPTION_TEXTS[2],
    'CON': ETIOLOGY_OPTION_TEXTS[7], 
}

def add_etpr_question(row, *, seed=None):
    rng = random.Random(seed if seed is not None else row.name)

    # 1️⃣ Randomly select a question variant
    row['question'] = rng.choice(etiology_question_variants)

    # 2️⃣ Shuffle options and assign fresh letters
    shuffled = ETIOLOGY_OPTION_TEXTS.copy()
    rng.shuffle(shuffled)
    letters = list(string.ascii_uppercase[:len(shuffled)])
    row['options'] = "\n".join(f"{ltr}. {opt}" for ltr, opt in zip(letters, shuffled))

    # 3️⃣ Get correct text from ETPR code (default to "Not applicable" if unknown)
    correct_text = ETPR_ANSWER_MAP.get(row['DX'])

    if correct_text is None:
        raise ValueError(f"Invalid ETPR value: {row['DX']}")


    # 4️⃣ Get ground truth letter from shuffled list
    if correct_text in shuffled:
        row['ground_truth'] = letters[shuffled.index(correct_text)]
    else:
        row['ground_truth'] = None  # shouldn't happen unless text mismatch
        
    row['ground_truth_text'] = correct_text

    return row


In [16]:
# Apply the function to the MCI training set
test_etpr = test.apply(add_etpr_question, axis=1)

In [17]:
test_etpr['ground_truth_text'].value_counts()

ground_truth_text
Frontotemporal lobar degeneration and its variants, including primary progressive aphasia, corticobasal degeneration and progressive supranuclear palsy, and with or without amyotrophic lateral sclerosis (FTLD)    156
Not applicable (no cognitive impairment)                                                                                                                                                                             136
Name: count, dtype: int64

In [18]:
test_etpr["ID"] = test_etpr["LONI_ID"]
test_etpr = test_etpr[columns]
print(len(test_etpr))
test_etpr = test_etpr.dropna(how='any').reset_index(drop=True)
print(len(test_etpr))

292
292


/scratch/ipykernel_223529/125116916.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_etpr["ID"] = test_etpr["LONI_ID"]


In [19]:
test_etpr['Q_TYPE'] = "Primary etiology"

In [20]:
test_etpr.head()

,ID,patient_summary,diag_summary,visit_summary,question,options,ground_truth,ground_truth_text,COHORT,Q_TYPE
0,1_S_0119,"{""Subject Demographics"": {""Gender"": ""Female"", ...","{""Diagnosis"": ""Behavioral Variant FTD""}",The patient is a 64-year-old white female with...,"Using the information provided, identify the p...",A. Not applicable (no cognitive impairment)\nB...,F,Frontotemporal lobar degeneration and its vari...,NIFD,Primary etiology
1,2_S_0007,"{""Subject Demographics"": {""Gender"": ""Male"", ""Y...","{""Diagnosis"": ""Behavioral Variant FTD""}",The subject is a 66-year-old White male with 1...,Identify the primary etiology of the patient's...,A. Psychiatric conditions including schizophre...,G,Frontotemporal lobar degeneration and its vari...,NIFD,Primary etiology
2,1_S_0059,"{""Subject Demographics"": {""Gender"": ""Female"", ...","{""Diagnosis"": ""Behavioral Variant FTD""}",The patient is a 48-year-old White female with...,Identify the primary etiologic diagnosis of th...,A. Vascular brain injury or vascular dementia ...,F,Frontotemporal lobar degeneration and its vari...,NIFD,Primary etiology
3,1_S_0248,"{""Subject Demographics"": {""Gender"": ""Male"", ""Y...","{""Diagnosis"": ""Healthy Control""}",The patient is a 75-year-old White male with 1...,Identify the primary etiology of the patient's...,A. Lewy body disease (LBD)\nB. Psychiatric con...,G,Not applicable (no cognitive impairment),NIFD,Primary etiology
4,1_S_0203,"{""Subject Demographics"": {""Gender"": ""Male"", ""Y...","{""Diagnosis"": ""Behavioral Variant FTD""}",The subject is a 62-year-old White male with 2...,Identify the primary etiology of the patient's...,A. Lewy body disease (LBD)\nB. Frontotemporal ...,B,Frontotemporal lobar degeneration and its vari...,NIFD,Primary etiology


In [21]:
test_etpr.to_csv(f"{directory_path}/training_data/testing_data_grpo/with_summary/test_etpr.csv", index=False)

# Add FTLD subtype question

In [22]:
import random, string

ftld_question_variants = [
    "Identify the patient's subtype of Frontotemporal Dementia using the information provided, if applicable.",
    "Based on the information provided, determine the subtype of Frontotemporal Dementia, if applicable.",
    "If applicable, use the provided information to identify the subtype of Frontotemporal Dementia.",
    "Determine the Frontotemporal Dementia subtype for the patient using the available information, if applicable.",
    "Using the information provided, classify the patient's Frontotemporal Dementia subtype, if applicable."
]


# Raw etiology answer texts in original order
FTLD_OPTION_TEXTS = [
    "Behavioral variant frontotemporal dementia",
    "Progressive non-fluent aphasia",
    "Semantic variant primary progressive aphasia",
    "Not applicable (no cognitive impairment)",
]

# Mapping of FTLD code to correct answer text
FTLD_ANSWER_MAP = {
    'BV': FTLD_OPTION_TEXTS[0],
    'PNFA': FTLD_OPTION_TEXTS[1],
    'SV': FTLD_OPTION_TEXTS[2],
    'CON': FTLD_OPTION_TEXTS[3],
}

def add_ftld_question(row, *, seed=None):
    rng = random.Random(seed if seed is not None else row.name)

    # 1️⃣ Randomly select a question variant
    row['question'] = rng.choice(ftld_question_variants)

    # 2️⃣ Shuffle options and assign fresh letters
    shuffled = FTLD_OPTION_TEXTS.copy()
    rng.shuffle(shuffled)
    letters = list(string.ascii_uppercase[:len(shuffled)])
    row['options'] = "\n".join(f"{ltr}. {opt}" for ltr, opt in zip(letters, shuffled))

    # 3️⃣ Get correct text from FTLD code (default to "Not applicable" if unknown)
    correct_text = FTLD_ANSWER_MAP.get(row['DX'])

    if correct_text is None:
        raise ValueError(f"Invalid FTLD value: {row['DX']}")


    # 4️⃣ Get ground truth letter from shuffled list
    if correct_text in shuffled:
        row['ground_truth'] = letters[shuffled.index(correct_text)]
    else:
        row['ground_truth'] = None  # shouldn't happen unless text mismatch
        
    row['ground_truth_text'] = correct_text

    return row


In [23]:
# Apply the function to the MCI training set
train_ftld = test.apply(add_ftld_question, axis=1)

In [24]:
train_ftld[train_ftld['ground_truth'].isna()]

,LONI_ID,CLINICAL_LINKDATE,DX,SITE,VISIT_NUMBER,DOB,GENDER,EDUCATION,RACE,CDR_DCDATE,...,COHORT,patient_summary,diag_summary,brain_findings_summary,visit_summary_prompt,visit_summary,question,options,ground_truth,ground_truth_text


In [25]:
train_ftld.iloc[0]['patient_summary']

'{"Subject Demographics": {"Gender": "Female", "Years of Education": 16, "Race": "White", "Age": 64}, "Unified Parkinson\'s Disease Rating Scale (UPDRS)": {"MD Exam - Unified Parkinson\'s Disease Rating Scale": 20}, "Neuropsychiatric Inventory": {"Neuropsychiatric Inventory - Q Form - Delusions": "No", "Neuropsychiatric Inventory - Q Form - Hallucination Distress": 0, "Neuropsychiatric Inventory - Q Form - Agitation": "No", "Neuropsychiatric Inventory - Q Form - Depression": "Yes", "Neuropsychiatric Inventory - Q Form - Depression Severity": 2, "Neuropsychiatric Inventory - Q Form - Anxiety": "No", "Neuropsychiatric Inventory - Q Form - Elation/Euphoria": "Yes", "Neuropsychiatric Inventory - Q Form - Elation/Euphoria Severity": 2, "Neuropsychiatric Inventory - Q Form - Apathy/Indifference": "Yes", "Neuropsychiatric Inventory - Q Form - Apathy/Indifference Severity": 1, "Neuropsychiatric Inventory - Q Form - Disinhibition": "Yes", "Neuropsychiatric Inventory - Q Form - Disinhibition Sev

In [26]:
train_ftld["ID"] = train_ftld["LONI_ID"]
train_ftld = train_ftld[columns]
print(len(train_ftld))
train_ftld = train_ftld.dropna(how='any').reset_index(drop=True)
print(len(train_ftld))

292
292


/scratch/ipykernel_223529/3783388533.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_ftld["ID"] = train_ftld["LONI_ID"]


In [27]:
print(train_ftld.iloc[-1]['diag_summary'])

{"Diagnosis": "Healthy Control"}


In [28]:
train_ftld['ground_truth_text'].value_counts()

ground_truth_text
Not applicable (no cognitive impairment)        136
Behavioral variant frontotemporal dementia       77
Progressive non-fluent aphasia                   40
Semantic variant primary progressive aphasia     39
Name: count, dtype: int64

In [29]:
train_ftld['Q_TYPE'] = "FTD subtype"

In [30]:
train_ftld.head()

,ID,patient_summary,diag_summary,visit_summary,question,options,ground_truth,ground_truth_text,COHORT,Q_TYPE
0,1_S_0119,"{""Subject Demographics"": {""Gender"": ""Female"", ...","{""Diagnosis"": ""Behavioral Variant FTD""}",The patient is a 64-year-old white female with...,Determine the Frontotemporal Dementia subtype ...,A. Semantic variant primary progressive aphasi...,C,Behavioral variant frontotemporal dementia,NIFD,FTD subtype
1,2_S_0007,"{""Subject Demographics"": {""Gender"": ""Male"", ""Y...","{""Diagnosis"": ""Behavioral Variant FTD""}",The subject is a 66-year-old White male with 1...,"Based on the information provided, determine t...",A. Semantic variant primary progressive aphasi...,D,Behavioral variant frontotemporal dementia,NIFD,FTD subtype
2,1_S_0059,"{""Subject Demographics"": {""Gender"": ""Female"", ...","{""Diagnosis"": ""Behavioral Variant FTD""}",The patient is a 48-year-old White female with...,Identify the patient's subtype of Frontotempor...,A. Semantic variant primary progressive aphasi...,D,Behavioral variant frontotemporal dementia,NIFD,FTD subtype
3,1_S_0248,"{""Subject Demographics"": {""Gender"": ""Male"", ""Y...","{""Diagnosis"": ""Healthy Control""}",The patient is a 75-year-old White male with 1...,"Based on the information provided, determine t...",A. Behavioral variant frontotemporal dementia\...,C,Not applicable (no cognitive impairment),NIFD,FTD subtype
4,1_S_0203,"{""Subject Demographics"": {""Gender"": ""Male"", ""Y...","{""Diagnosis"": ""Behavioral Variant FTD""}",The subject is a 62-year-old White male with 2...,"Based on the information provided, determine t...",A. Not applicable (no cognitive impairment)\nB...,C,Behavioral variant frontotemporal dementia,NIFD,FTD subtype


In [31]:
train_ftld.to_csv(f"{directory_path}/training_data/testing_data_grpo/with_summary/test_ftld.csv", index=False)